# Übung 3: Welche Aufträge werden scheitern?
**Internet of Production – KI, Woche 1** · ca. 25 min · Gruppen mit 2–3 Personen

**Szenario:** Sie sind Leiter der Vorfertigung. Vor dem Start eines Auftrags möchten Sie wissen, ob er voraussichtlich scheitert. 11,5 % der Aufträge scheitern, und jede Störung kostet Druckerzeit und Material.

**Ziel:** Nach der Frühwarnung für den Drucker (Übung 2) betrachten Sie nun die **Aufträge**. Trainieren Sie Klassifikatoren auf den Auftragsdaten, beurteilen Sie sie mit den richtigen Metriken und geben Sie dem Vorfertigungsleiter drei Sätze Empfehlung. Jedes `___` ist eine Lücke.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

DATA = Path('../../00_Data/printfarm')

## Schritt 1: Aufträge laden und Ziel definieren

In [ ]:
orders = pd.read_csv(DATA / 'Printproduction.csv', sep=';', decimal=',').dropna(how='all')
orders['Start'] = pd.to_datetime(orders['Start'], format='%d.%m.%Y %H:%M')
orders['Material qty. in g'] = pd.to_numeric(orders['Material qty. in g'].astype(str).str.replace(',', '.'), errors='coerce')
orders = orders.sort_values('Start').reset_index(drop=True)
orders['failed'] = (orders['Status'] == ___).astype(int)
print(orders.shape)
orders.head()

## Schritt 2: Welche Spalten sind bei **Auftragsstart** bekannt?
Betrachten Sie die Spalten. Wählen Sie nur die aus, die der Vorfertigungsleiter **vor** dem Start des Auftrags kennt. Überlegen Sie je Spalte: Könnten Sie ihren Wert zum Zeitpunkt der Freigabe ablesen?

In [ ]:
orders['hour'] = orders['Start'].dt.hour
orders['weekday'] = orders['Start'].dt.dayofweek
cat = ['Printer', 'BOM Item', 'Colour']          # categorical features
num = ['Lot size', 'Target Duration', 'hour', 'weekday']   # numeric features
# TODO: check the list. Which columns of `orders` are known at order start? Remove the ones that are not.
features = cat + num
print(orders[features].dtypes)

## Schritt 3: Nach Zeit aufteilen
Das Modell wird auf **zukünftige** Aufträge angewendet. Trainieren Sie mit den ersten 70 % der Aufträge (nach Startzeit) und testen Sie mit den letzten 30 %.

In [ ]:
cut = int(len(orders) * ___)
train, test = orders.iloc[:cut], orders.iloc[cut:]
X_train, y_train = train[features], train['failed']
X_test, y_test = test[features], test['failed']
print('train:', len(train), 'orders, failed:', y_train.sum())
print('test :', len(test), 'orders, failed:', y_test.sum())

## Schritt 4: Baseline
Welche Accuracy erreicht ein Modell, das **immer `completed` vorhersagt**? Wie hoch sind Precision und Recall?

In [ ]:
def report(name, y_true, y_pred):
    print(f'{name:28s} accuracy {accuracy_score(y_true, y_pred):6.1%}   recall {recall_score(y_true, y_pred, zero_division=0):6.1%}   precision {precision_score(y_true, y_pred, zero_division=0):6.1%}')

base = DummyClassifier(strategy=___).fit(X_train, y_train)
report('baseline: always completed', y_test, base.predict(X_test))

## Schritt 5: Entscheidungsbaum
Kategoriale Spalten werden one-hot kodiert. Variieren Sie die Tiefe des Baums.

In [ ]:
def make_tree(depth):
    prep = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat)], remainder='passthrough')
    return make_pipeline(prep, DecisionTreeClassifier(max_depth=depth, class_weight='balanced', random_state=0))

for depth in [___]:        # try several depths
    m = make_tree(depth).fit(X_train, y_train)
    report(f'tree, depth {depth}', y_test, m.predict(X_test))

## Schritt 6 (optional, bei Zeitmangel überspringen): KNN
KNN benötigt skalierte numerische Merkmale.

In [ ]:
def make_knn(k):
    prep = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat), ('num', StandardScaler(), num)])
    return make_pipeline(prep, KNeighborsClassifier(n_neighbors=k))

for k in [___]:
    m = make_knn(k).fit(X_train, y_train)
    report(f'KNN, k = {k}', y_test, m.predict(X_test))

## Schritt 6: Konfusionsmatrix Ihres besten Modells
Wählen Sie das Modell, das Sie für den Vorfertigungsleiter am nützlichsten finden, und begründen Sie die Wahl mit einer Metrik.

In [ ]:
best = make_tree(___).fit(X_train, y_train)      # your choice
pred = best.predict(X_test)
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=['completed', 'failed']).plot(cmap='Blues')
plt.title('Your model'); plt.show()

## Schritt 8: Ist eine einfache Regel genauso gut?
Berechnen Sie die Ausfallrate je Bauteil und testen Sie die Regel *„Teile mit der höchsten Ausfallrate scheitern“*.

In [ ]:
rate = orders.groupby('BOM Item')['failed'].agg(['mean', 'sum', 'size']).sort_values('mean', ascending=False)
rate['mean'] = (rate['mean'] * 100).round(1)
print(rate.head(6))

risky = list(rate.index[:___])          # how many parts?
rule = test['BOM Item'].isin(risky).astype(int)
report(f'rule: parts {risky} fail', y_test, rule)

## Schritt 9: Und wenn wir auch Spalten von *nach* dem Auftrag nutzen?
Führen Sie die Zelle aus. Die Ergebnisse sehen hervorragend aus. **Warum dürfen Sie ihnen nicht trauen?**

In [ ]:
leaky = num + ['Actual Duration', 'Material qty. in g']
prep = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat)], remainder='passthrough')
m = make_pipeline(prep, DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=0)).fit(train[cat + leaky], y_train)
report('tree WITH Actual Duration, qty', y_test, m.predict(test[cat + leaky]))
print()
print(orders.groupby('Status')[['Actual Duration', 'Material qty. in g']].median())

**Warum ist dieses Ergebnis nicht nutzbar?**

*Ihre Antwort:*

## Schritt 10: Warum scheitern Aufträge? Aufträge mit Störungen verknüpfen
Eine Druckerstörung wird mit Zeitstempel protokolliert. Prüfen Sie für jeden Auftrag, ob zwischen Start und Ende des Auftrags eine Störung **seines Druckers** aufgetreten ist.

In [ ]:
def load_errors(k):
    e = pd.read_csv(DATA / f'Printer_{k}_error.csv', sep=';').dropna(how='all')
    e['Timestamp'] = pd.to_datetime(e['Timestamp'], format='%d.%m.%Y %H:%M')
    e['Printer'] = k
    return e

errors = pd.concat([load_errors(k) for k in (0, 1, 2)])
orders['End'] = orders['Start'] + pd.to_timedelta(orders['Actual Duration'], unit='m')

def disturbance(o):
    tol = pd.Timedelta(minutes=5)
    e = errors[(errors['Printer'] == o['Printer']) & (errors['Timestamp'] >= o['Start'] - tol) & (errors['Timestamp'] <= o['End'] + tol)]
    return e['Error'].iloc[0] if len(e) else 'none'

orders['disturbance'] = orders.apply(disturbance, axis=1)
print(pd.crosstab(orders[___], orders['disturbance'] != 'none', colnames=['disturbance during the order']))
print()
print(pd.crosstab(orders.loc[orders['failed'] == 1, 'Printer'], orders.loc[orders['failed'] == 1, 'disturbance']))

**Was bedeutet das für die Vorhersage gescheiterter Aufträge, und wie hängt es mit Übung 2 zusammen?**

*Ihre Antwort:*

## Ihre Empfehlung (drei Sätze)
Was soll der Vorfertigungsleiter tun? Welches Modell oder welche Regel, welche Metrik, und was fehlt noch?

*Ihre Antwort:*